In [ ]:
!pip install tensorflow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import AdamW

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
tf.keras.mixed_precision.set_global_policy('mixed_float16')

base_dir = "/content/drive/MyDrive/Simon Says Game/Emotions"
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")
model_save_path = os.path.join(base_dir, "emotion_model_grayscale_t4.h5")

BATCH_SIZE = 256

train_datagen = ImageDataGenerator(
    rescale=1./255, validation_split=0.15,
    rotation_range=15,
    width_shift_range=0.12,
    height_shift_range=0.12,
    shear_range=0.12,
    zoom_range=0.12,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15]  # Perfect for grayscale
)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(48,48),
    color_mode='grayscale',
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', shuffle=True
)

validation_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(48,48),
    color_mode='grayscale',
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', shuffle=True
)

test_generator = ImageDataGenerator(rescale=1./255).flow_from_directory(
    test_dir, target_size=(48,48), color_mode='grayscale',
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

num_classes = train_generator.num_classes
print(f"✅ Grayscale dataset loaded: {num_classes} classes")


model = Sequential([
    Input(shape=(48, 48, 1)),

    # Edge detection optimized for grayscale
    Conv2D(32, (5,5), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2), Dropout(0.2),

    # Facial features
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2), Dropout(0.25),

    # Fine details (eyes, mouth)
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2), Dropout(0.3),

    # Global features
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    GlobalAveragePooling2D(),

    # Emotion classifier
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(num_classes, activation='softmax', dtype='float32')
])

model.compile(
    optimizer=AdamW(learning_rate=0.001, weight_decay=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.4, patience=4, verbose=1, mode='max'),
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, mode='max'),
    ModelCheckpoint(model_save_path, monitor='val_accuracy', save_best_only=True, mode='max')
]

print("🚀 GRAYSCALE T4 Training START (72-78% expected)")
print(f"📊 Batches/epoch: {len(train_generator)}")

history = model.fit(
    train_generator,
    epochs=40,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

# Test
test_loss, test_acc = model.evaluate(test_generator)
print(f"🏆 FINAL TEST ACCURACY: {test_acc:.4f}")

model.save(model_save_path)
print(f"✅ Grayscale T4 Model saved!")

Found 24417 images belonging to 7 classes.
Found 4305 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.
✅ Grayscale dataset loaded: 7 classes
🚀 GRAYSCALE T4 Training START (72-78% expected)
📊 Batches/epoch: 96
Epoch 1/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 13s/step - accuracy: 0.1743 - loss: 2.5729 

96/96 ━━━━━━━━━━━━━━━━━━━━ 1795s 18s/step - accuracy: 0.1930 - loss: 2.3667 - val_accuracy: 0.2511 - val_loss: 1.9358 - learning_rate: 0.0010
Epoch 2/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 878ms/step - accuracy: 0.2503 - loss: 2.0531

96/96 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.2644 - loss: 1.9843 - val_accuracy: 0.2513 - val_loss: 2.3014 - learning_rate: 0.0010
Epoch 3/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 98s 1s/step - accuracy: 0.3469 - loss: 1.7334 - val_accuracy: 0.2513 - val_loss: 2.6296 - learning_rate: 0.0010
Epoch 4/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 863ms/step - accuracy: 0.3965 - loss: 1.6181

96/96 ━━━━━━━━━━━━━━━━━━━━ 98s 1s/step - accuracy: 0.4073 - loss: 1.5923 - val_accuracy: 0.2815 - val_loss: 1.8487 - learning_rate: 0.0010
Epoch 5/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 869ms/step - accuracy: 0.4400 - loss: 1.4926

96/96 ━━━━━━━━━━━━━━━━━━━━ 98s 1s/step - accuracy: 0.4439 - loss: 1.4930 - val_accuracy: 0.3280 - val_loss: 1.9731 - learning_rate: 0.0010
Epoch 6/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 854ms/step - accuracy: 0.4494 - loss: 1.4834

...

96/96 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.6427 - loss: 0.9445 - val_accuracy: 0.6369 - val_loss: 0.9796 - learning_rate: 4.0000e-04
Epoch 31/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 96s 999ms/step - accuracy: 0.6484 - loss: 0.9303 - val_accuracy: 0.6256 - val_loss: 0.9910 - learning_rate: 4.0000e-04
Epoch 32/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 830ms/step - accuracy: 0.6493 - loss: 0.9326

96/96 ━━━━━━━━━━━━━━━━━━━━ 94s 982ms/step - accuracy: 0.6545 - loss: 0.9190 - val_accuracy: 0.6390 - val_loss: 0.9944 - learning_rate: 4.0000e-04
Epoch 33/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 95s 990ms/step - accuracy: 0.6572 - loss: 0.9191 - val_accuracy: 0.6172 - val_loss: 1.0094 - learning_rate: 4.0000e-04
Epoch 34/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 96s 1000ms/step - accuracy: 0.6556 - loss: 0.9087 - val_accuracy: 0.6341 - val_loss: 0.9856 - learning_rate: 4.0000e-04
Epoch 35/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 95s 988ms/step - accuracy: 0.6606 - loss: 0.9090 - val_accuracy: 0.6279 - val_loss: 0.9872 - learning_rate: 4.0000e-04
Epoch 36/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 848ms/step - accuracy: 0.6625 - loss: 0.8994
Epoch 36: ReduceLROnPlateau reducing learning rate to 0.00016000000759959222.
96/96 ━━━━━━━━━━━━━━━━━━━━ 96s 999ms/step - accuracy: 0.6600 - loss: 0.9004 - val_accuracy: 0.6328 - val_loss: 0.9903 - learning_rate: 4.0000e-04
Epoch 37/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 95s 994ms/step - accuracy: 0.

96/96 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - accuracy: 0.6662 - loss: 0.8874 - val_accuracy: 0.6458 - val_loss: 0.9603 - learning_rate: 1.6000e-04
Epoch 39/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 95s 994ms/step - accuracy: 0.6688 - loss: 0.8729 - val_accuracy: 0.6432 - val_loss: 0.9630 - learning_rate: 1.6000e-04
Epoch 40/40
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 844ms/step - accuracy: 0.6749 - loss: 0.8698

96/96 ━━━━━━━━━━━━━━━━━━━━ 96s 997ms/step - accuracy: 0.6727 - loss: 0.8738 - val_accuracy: 0.6462 - val_loss: 0.9674 - learning_rate: 1.6000e-04
29/29 ━━━━━━━━━━━━━━━━━━━━ 2017s 72s/step - accuracy: 0.6541 - loss: 0.9403


🏆 FINAL TEST ACCURACY: 0.6541
✅ Grayscale T4 Model saved!
